# Queries, tables and conditions

A parameter sweep leaves many records of one type. This notebook finds them by spec with
`Record.query`, lists them as a DataFrame with `Collection.table` and `Record.info`, filters
records, tables and xarray results with one condition grammar (`td.where`, `td.any_of`,
`td.exactly`), compares specs with `td.diff`, and maintains the root with the index and the
`tensordir` command line.

In [ ]:
import os
import re
import subprocess
import sys
import tempfile
from pathlib import Path

import numpy as np
import xarray as xr

import tensordir as td

root = Path(tempfile.mkdtemp(prefix="tensordir-queries-"))
td.set_root(root)

## A sweep

`Net` trains a toy network; its settings include a nested optimizer spec and a list-valued
field (`layers`). One configuration fails on purpose: tensordir stores the failure with its traceback.

In [ ]:
class Optimizer(td.Spec):
    lr: float = 1e-2
    momentum: float = 0.0


class Net(td.Record):
    def __init__(
        self,
        /,
        *,
        layers: list[int] = [8],  # copied per call: list defaults are safe here
        activation: str = "relu",
        optimizer: Optimizer = Optimizer(),
        seed: int = 0,
        tag: str | None = None,
    ):
        if optimizer.lr > 0.5:
            raise FloatingPointError("loss diverged")
        rng = np.random.default_rng(seed)
        steps = np.arange(50)
        self.loss = np.exp(-optimizer.lr * 10 * steps / sum(layers) * 8) + 0.01 * rng.random(50)
        self.summary = {"final_loss": float(self.loss[-1])}


for layers in ([8], [8, 16], [16]):
    for lr in (1e-3, 1e-2, 1e-1):
        for seed in (0, 1):
            Net(layers=layers, optimizer=Optimizer(lr=lr), seed=seed)
Net(activation="tanh", tag="baseline-tanh")
Net(activation="gelu", tag="probe")

try:
    Net(optimizer=Optimizer(lr=1.0))
except td.BuildFailed as exc:
    print("stored failure:", exc.error["type"], exc.error["message"])

## Query records by spec

`Record.query(**conditions)` returns a `Collection` of the type's records whose flattened spec
meets every condition. Nested fields use `__` (`optimizer__lr`). Only finished records are
returned unless `status=` says otherwise (`status=None`: all).

In [ ]:
done = Net.query()
print(done)
print(Net.query(status=None).status_counts())
print(Net.query(status="failed"))

In [ ]:
Net.query(optimizer__lr=1e-2, seed=0)  # the notebook repr is a table of what varies

## Conditions

The same conditions work in `Record.query`, `Record.info`, `Collection.where`, `td.where` and the
`.td.where` accessors:

| condition | matches |
|---|---|
| `0.1`, `"relu"` | equal values; floats within a relative tolerance of 1e-9 |
| `None` | missing values (None, NaN, NA) |
| `">1e-3, <0.1"` | values meeting every comparison (comma means AND) |
| `[0, 1]`, `(8, 16)` | values equal to the whole list, or to any of its items |
| `{0, 1}` | values equal to any item |
| `td.any_of(0, 1)` | values meeting any item; items are conditions; never the whole list |
| `td.exactly([8, 16])` | values equal to the whole list only; strings are not parsed |
| `re.compile("^base")` | values whose string form contains a match |
| `lambda v: v > 0` | values for which the callable returns a truthy result |

In [ ]:
def count(**conditions):
    return len(Net.query(**conditions))


print("lr == 1e-2                 ", count(optimizer__lr=1e-2))
print("1e-3 < lr <= 1e-1          ", count(optimizer__lr=">1e-3, <=1e-1"))
print("seed in [0, 1]             ", count(seed=[0, 1]))
print("activation in {tanh, gelu} ", count(activation={"tanh", "gelu"}))
print("tag is missing             ", count(tag=None))
print("tag starts with 'base'     ", count(tag=re.compile("^base")))
print("lr > 5e-3 (callable)       ", count(optimizer__lr=lambda lr: lr > 5e-3))
print("lr == 1e-3 or lr > 5e-2    ", count(optimizer__lr=td.any_of(1e-3, ">5e-2")))

### List-valued fields: `any_of` and `exactly`

A bare list is tried as a whole first, then item by item. For a scalar field like `seed`,
`[0, 1]` means "any of". For the list field `layers`, `[8, 16]` means "equal to `[8, 16]`",
and `[[8], [16]]` means "equal to `[8]` or to `[16]`". `td.exactly` and `td.any_of` allow only
one reading, which makes code that builds conditions from variables unambiguous.

In [ ]:
print("layers=[8, 16]                ", count(layers=[8, 16]))
print("layers=td.exactly([8, 16])    ", count(layers=td.exactly([8, 16])))
print("layers=[[8], [16]]            ", count(layers=[[8], [16]]))
print("layers=td.any_of([8], [16])   ", count(layers=td.any_of([8], [16])))
print("layers=td.any_of(8, 16)       ", count(layers=td.any_of(8, 16)), "(items are compared whole: no list equals 8)")

`td.exactly` also compares strings literally, where a bare string would be parsed as a
comparison.

In [ ]:
Net(tag="<v2")
print(count(tag="<v2"), "tags sort before 'v2';", count(tag=td.exactly("<v2")), "tag is the string '<v2'")

## Collections

A `Collection` is an ordered, sliceable sequence of records. `where` filters it by a predicate
on the record, by conditions, or by a mapping with dotted names.

In [ ]:
sweep = Net.query(layers=[8])
low_loss = sweep.where(lambda record: record.summary["final_loss"] < 0.5)
print(len(sweep), len(low_loss), len(sweep.where({"optimizer.lr": 1e-1})))
print(sweep[:2], [key[:16] for key in sweep[:2].keys()])
print(td.where(sweep, seed=1).keys() == sweep.where(seed=1).keys())  # the function form

## Tables

`Collection.table()` gives one row per record: type, key, status, timestamps, duration, size,
number of fields, path and one `spec.<dotted name>` column per spec field.

In [ ]:
table = Net.query().table()
print(list(table.columns))

`varying_only=True` keeps the spec columns that differ between the records. `depth=k` keeps spec
columns nested at most `k` levels deep (`depth=1`: top-level fields only), and `include`
regexes bring back others. `sizes=False` skips walking each record directory.

In [ ]:
Net.query(seed=0).table(sizes=False, varying_only=True).filter(like="spec.")

In [ ]:
Net.query(seed=0).table(sizes=False, depth=1, include=[r"optimizer\.lr"]).filter(like="spec.").head()

`Record.info(...)` is `Record.query(status=None, ...).table(...)` in one call, so failed
records appear too. On a record, `record.info()` is its one-row table.

In [ ]:
info = Net.info(depth=1, include=["lr$"], sizes=False, activation="relu", layers=[8])
info[["key", "status", "duration_s", "spec.optimizer.lr", "spec.seed"]]

In [ ]:
Net.query(tag="probe")[0].info(sizes=False)[["key", "status", "n_fields", "spec.activation", "spec.tag"]]

Tables are ordinary DataFrames. `td.where` filters their rows with the same conditions; a
condition name finds the column `name`, else `spec.name`, else an index level.

In [ ]:
table = Net.query().table(sizes=False)
best = td.where(table, layers=td.any_of([8], [16]), optimizer__lr=">=1e-2")
best.groupby("spec.optimizer.lr").size()

In [ ]:
table.pipe(td.where, status="done", seed=1, tag=None).shape

## Conditions on xarray results

`td.where` filters xarray objects on their coordinates, as flyvis's `.custom.where` did.
`Collection.map(fn, dim=, coords=)` stacks one xarray result per record along a new dimension
with chosen spec fields as coordinates (see the notebook on collections and views).

In [ ]:
losses = Net.query(layers=[8]).map(
    lambda record: xr.DataArray(record.loss, dims="step"), dim="net", coords=["optimizer.lr", "seed"]
)
fast = td.where(losses, **{"optimizer.lr": ">1e-3"}, seed=0)
print(losses.sizes, "->", fast.sizes)
fast.coords

## Compare specs

`td.diff(a, b)` compares two specs, records, views or dicts by dotted name: changed, added and
removed fields. `hashed=True` keeps only fields that enter the key.

In [ ]:
a, b = Net.query(layers=[8], seed=0)[:2]
d = td.diff(a, b)
print(d)
print(d.changed, bool(td.diff(a, a)))

## Reruns and cleanup

Failed records raise `BuildFailed` with the stored traceback on every call, until they are
deleted or rebuilt with `retry_failed=True`.

In [ ]:
failed = Net.query(status="failed")
print(failed[0].status, failed[0].error["message"])
print(failed.delete(), "deleted;", Net.query(status=None).status_counts())

## The index and the command line

Queries scan `_meta.json` files. `td.reindex()` writes `.tensordir/index.parquet`, which
queries trust per type directory while that directory's modification time is unchanged:
records enter and leave by `rename`, which updates it.

In [ ]:
print(td.reindex().relative_to(root))
print(len(Net.query()))

The `tensordir` command lists records (`ls`), counts them by status (`status`), rebuilds the
index (`reindex`) and removes stale leases, orphaned temporary builds and trash (`gc`). It
reads the base root from `$TENSORDIR_ROOT` or `--root`.

In [ ]:
env = {**os.environ, "TENSORDIR_ROOT": str(root)}


def tensordir(*args, lines=5):
    """Run the `tensordir` command on this notebook's root and print the first `lines` lines."""
    result = subprocess.run([sys.executable, "-m", "tensordir.cli", *args], env=env, capture_output=True, text=True)
    print("\n".join((result.stdout or result.stderr).splitlines()[:lines]) or "(no output)")


tensordir("status")
tensordir("ls", "Net", "--status", "done")
tensordir("gc", "--dry-run")  # nothing to clean up; see the notebook on concurrent builds